# 02 — Graph Construction

Build the heterogeneous PyG graph from all four datasets. Create train/val/test splits and cold-start split. Save everything to disk.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import os
from collections import Counter

from src.data_loading import load_all
from src.preprocessing import (
    prefix_chg_genes, build_id_maps, build_all_edges,
    save_processed, load_processed
)
from src.graph_builder import build_hetero_data, build_homo_data
from src.utils import set_seed, DATA_SPLITS, DATA_PROCESSED

set_seed(42)

## 1. Load & Prepare DataFrames

In [ ]:
dfs = load_all()
ddi = dfs['ChCh-Miner']
dtg = dfs['ChG-Miner']
dga = dfs['DG-AssocMiner']
dda = dfs['DCh-Miner']

# Prefix ChG-Miner genes with UNIPROT: to distinguish from ENTREZ: genes
dtg = prefix_chg_genes(dtg)
print(f"Sample DTG gene IDs after prefixing: {dtg['gene'].unique()[:3]}")

## 2. Build ID Maps

In [ ]:
id_maps, stats = build_id_maps(ddi, dtg, dga, dda)

print("=== Graph Statistics ===")
for k, v in stats.items():
    print(f"  {k}: {v:,}")

print(f"\nNode type sizes:")
for ntype, nmap in id_maps.items():
    print(f"  {ntype}: {len(nmap):,} nodes")

## 3. Build Edge Indices

In [ ]:
edges = build_all_edges(ddi, dtg, dga, id_maps, dda)

print("\n=== Edge Statistics ===")
total_edges = 0
for name, ei in edges.items():
    print(f"  {name}: {ei.shape[1]:,} edges")
    total_edges += ei.shape[1]
print(f"  TOTAL: {total_edges:,} edges")

## 4. Save Processed Data

In [ ]:
save_processed(id_maps, edges, stats)

## 5. Build PyG Objects

In [ ]:
hetero_data = build_hetero_data(id_maps, edges)
print("HeteroData object:")
print(hetero_data)

homo_ei, num_drugs = build_homo_data(id_maps, edges)
print(f"\nHomogeneous DDI graph: {num_drugs} nodes, {homo_ei.shape[1]} edges")

## 6. Train / Val / Test Split (DDI Edges Only)

80/10/10 split of the undirected DDI edges. Auxiliary edges remain in the graph for all phases.

In [ ]:
ddi_ei = edges['drug_drug']  # already undirected

# De-duplicate to get canonical edges (src < dst)
mask = ddi_ei[0] < ddi_ei[1]
canonical = ddi_ei[:, mask]
n_edges = canonical.shape[1]
print(f"Canonical (undirected) DDI edges: {n_edges:,}")

# Shuffle and split
perm = np.random.permutation(n_edges)
n_train = int(0.8 * n_edges)
n_val = int(0.1 * n_edges)

train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

train_edges = canonical[:, train_idx]
val_edges = canonical[:, val_idx]
test_edges = canonical[:, test_idx]

# Make each split bidirectional for message passing
def make_undirected(ei):
    return np.concatenate([ei, np.stack([ei[1], ei[0]])], axis=1)

train_edges_ud = make_undirected(train_edges)
val_edges_ud = make_undirected(val_edges)
test_edges_ud = make_undirected(test_edges)

print(f"Train: {train_edges.shape[1]:,} canonical ({train_edges_ud.shape[1]:,} directed)")
print(f"Val:   {val_edges.shape[1]:,} canonical ({val_edges_ud.shape[1]:,} directed)")
print(f"Test:  {test_edges.shape[1]:,} canonical ({test_edges_ud.shape[1]:,} directed)")

## 7. Cold-Start Split

Select ~10% of drugs (30th–60th percentile by degree) and remove ALL their DDI edges from training.

In [ ]:
# Drug degrees in the full DDI graph
drug_degrees = np.zeros(num_drugs, dtype=int)
for i in range(ddi_ei.shape[1]):
    drug_degrees[ddi_ei[0, i]] += 1
# Divide by 2 since undirected edges are stored twice
drug_degrees = drug_degrees // 2

p30 = np.percentile(drug_degrees[drug_degrees > 0], 30)
p60 = np.percentile(drug_degrees[drug_degrees > 0], 60)
print(f"Degree percentiles: P30={p30:.0f}, P60={p60:.0f}")

cold_start_candidates = np.where((drug_degrees >= p30) & (drug_degrees <= p60))[0]
np.random.shuffle(cold_start_candidates)
n_cold = max(1, int(0.1 * num_drugs))
cold_start_drugs = set(cold_start_candidates[:n_cold].tolist())
print(f"Cold-start drugs: {len(cold_start_drugs)} (target: {n_cold})")

# Split canonical edges into cold-start held-out vs warm training
cold_mask = np.array([
    canonical[0, i] in cold_start_drugs or canonical[1, i] in cold_start_drugs
    for i in range(canonical.shape[1])
])
cold_edges = canonical[:, cold_mask]
warm_edges = canonical[:, ~cold_mask]

# Further split warm edges into train/val/test
n_warm = warm_edges.shape[1]
perm_w = np.random.permutation(n_warm)
n_w_train = int(0.85 * n_warm)
n_w_val = int(0.075 * n_warm)

cs_train = warm_edges[:, perm_w[:n_w_train]]
cs_val = warm_edges[:, perm_w[n_w_train:n_w_train + n_w_val]]
cs_test_warm = warm_edges[:, perm_w[n_w_train + n_w_val:]]

print(f"\nCold-start split:")
print(f"  Held-out cold edges:  {cold_edges.shape[1]:,}")
print(f"  Warm train edges:     {cs_train.shape[1]:,}")
print(f"  Warm val edges:       {cs_val.shape[1]:,}")
print(f"  Warm test edges:      {cs_test_warm.shape[1]:,}")

## 8. Save All Splits

In [ ]:
os.makedirs(DATA_SPLITS, exist_ok=True)

# Standard split
torch.save(torch.from_numpy(train_edges).long(), os.path.join(DATA_SPLITS, 'train_edges.pt'))
torch.save(torch.from_numpy(val_edges).long(), os.path.join(DATA_SPLITS, 'val_edges.pt'))
torch.save(torch.from_numpy(test_edges).long(), os.path.join(DATA_SPLITS, 'test_edges.pt'))
torch.save(torch.from_numpy(train_edges_ud).long(), os.path.join(DATA_SPLITS, 'train_edges_ud.pt'))

# Cold-start split
torch.save(torch.tensor(sorted(cold_start_drugs)), os.path.join(DATA_SPLITS, 'cold_start_drugs.pt'))
torch.save(torch.from_numpy(cold_edges).long(), os.path.join(DATA_SPLITS, 'cold_edges.pt'))
torch.save(torch.from_numpy(cs_train).long(), os.path.join(DATA_SPLITS, 'cs_train_edges.pt'))
torch.save(torch.from_numpy(cs_val).long(), os.path.join(DATA_SPLITS, 'cs_val_edges.pt'))
torch.save(torch.from_numpy(cs_test_warm).long(), os.path.join(DATA_SPLITS, 'cs_test_warm_edges.pt'))

print(f"All splits saved to {DATA_SPLITS}")
print(f"Files: {os.listdir(DATA_SPLITS)}")

## 9. Leakage Validation

In [ ]:
def edge_set(ei):
    return set(zip(ei[0].tolist(), ei[1].tolist()))

train_set = edge_set(train_edges)
val_set = edge_set(val_edges)
test_set = edge_set(test_edges)

print("Leakage check (standard split):")
print(f"  Train ∩ Val:  {len(train_set & val_set)} (should be 0)")
print(f"  Train ∩ Test: {len(train_set & test_set)} (should be 0)")
print(f"  Val ∩ Test:   {len(val_set & test_set)} (should be 0)")

# Cold-start: ensure no cold-start drug edges in training
cs_train_set = edge_set(cs_train)
cold_in_train = sum(
    1 for s, d in cs_train_set
    if s in cold_start_drugs or d in cold_start_drugs
)
print(f"\nCold-start leakage: {cold_in_train} cold-drug edges in training (should be 0)")